In [1]:
pip install ultralytics

Note: you may need to restart the kernel to use updated packages.


In [2]:
import cv2
import numpy as np
import math
from collections import defaultdict, deque
from ultralytics import YOLO

In [3]:
# Load the YOLO model
model = YOLO('yolo11l.pt')

In [4]:
class_list = model.names
class_list

{0: 'person',
 1: 'bicycle',
 2: 'car',
 3: 'motorcycle',
 4: 'airplane',
 5: 'bus',
 6: 'train',
 7: 'truck',
 8: 'boat',
 9: 'traffic light',
 10: 'fire hydrant',
 11: 'stop sign',
 12: 'parking meter',
 13: 'bench',
 14: 'bird',
 15: 'cat',
 16: 'dog',
 17: 'horse',
 18: 'sheep',
 19: 'cow',
 20: 'elephant',
 21: 'bear',
 22: 'zebra',
 23: 'giraffe',
 24: 'backpack',
 25: 'umbrella',
 26: 'handbag',
 27: 'tie',
 28: 'suitcase',
 29: 'frisbee',
 30: 'skis',
 31: 'snowboard',
 32: 'sports ball',
 33: 'kite',
 34: 'baseball bat',
 35: 'baseball glove',
 36: 'skateboard',
 37: 'surfboard',
 38: 'tennis racket',
 39: 'bottle',
 40: 'wine glass',
 41: 'cup',
 42: 'fork',
 43: 'knife',
 44: 'spoon',
 45: 'bowl',
 46: 'banana',
 47: 'apple',
 48: 'sandwich',
 49: 'orange',
 50: 'broccoli',
 51: 'carrot',
 52: 'hot dog',
 53: 'pizza',
 54: 'donut',
 55: 'cake',
 56: 'chair',
 57: 'couch',
 58: 'potted plant',
 59: 'bed',
 60: 'dining table',
 61: 'toilet',
 62: 'tv',
 63: 'laptop',
 64: 'mou

In [5]:
cap = cv2.VideoCapture(r"C:\Users\KIIT\Desktop\TrafficBrain AI\demo_count_vehicles\test_videos\4.mp4")
print(cap.isOpened())

True


## Speed Zone Configuration
Colour rules and class-specific speed limits based on **City Road** type.

In [6]:
# ── Road type: City Road ────────────────────────────────────────────
# Speed limits differ by vehicle class (from Speed Limit Guide)
SPEED_LIMITS = {
    'car':        50,   # km/h
    'motorcycle': 50,
    'bus':        40,
    'truck':      40,
}
DEFAULT_SPEED_LIMIT = 50  # fallback for any unlisted class

PIXELS_PER_METRE = 8.0   # calibrate this for your video

# ── Speed Zone Colour Rules ──────────────────────────────────────────
# BLUE   →  0–10 km/h   : very slow / stopped
# GREEN  → 11–60 km/h   : safe speed
# YELLOW → 61–class limit: approaching limit, caution
# RED    → above limit   : violation / speeding

def get_speed_color(speed, class_name):
    """
    Return BGR colour based on speed zone rules.
      Blue   : 0–10 km/h   (very slow / stopped)
      Green  : 11–60 km/h  (safe)
      Yellow : 61 km/h up to this vehicle's class speed limit (caution)
      Red    : above class speed limit (violation)
    """
    limit = SPEED_LIMITS.get(class_name, DEFAULT_SPEED_LIMIT)
    if speed <= 10:
        return (255, 100, 0)    # BLUE
    elif speed <= 60:
        return (0, 200, 0)      # GREEN
    elif speed <= limit:
        return (0, 220, 255)    # YELLOW
    else:
        return (0, 0, 255)      # RED  — violation

def is_violation(speed, class_name):
    """True when vehicle exceeds its class speed limit."""
    return speed > SPEED_LIMITS.get(class_name, DEFAULT_SPEED_LIMIT)

print("Speed config loaded  —  Road type: City Road")
print()
print("  Class speed limits:")
for cls, lim in SPEED_LIMITS.items():
    print(f"    {cls:<12}: {lim} km/h")
print()
print("  Colour zones:")
print("    BLUE   →  0–10 km/h  (very slow / stopped)")
print("    GREEN  → 11–60 km/h  (safe)")
print("    YELLOW → 61 km/h – class limit  (caution)")
print("    RED    → above class limit  (violation)")

Speed config loaded  —  Road type: City Road

  Class speed limits:
    car         : 50 km/h
    motorcycle  : 50 km/h
    bus         : 40 km/h
    truck       : 40 km/h

  Colour zones:
    BLUE   →  0–10 km/h  (very slow / stopped)
    GREEN  → 11–60 km/h  (safe)
    YELLOW → 61 km/h – class limit  (caution)
    RED    → above class limit  (violation)


## Speed Helper Function

In [7]:
def estimate_speed(positions, fps):
    """
    Calculate speed in km/h from last 2 tracked positions.
    positions: deque of (cx, cy, frame_idx) tuples
    """
    if len(positions) < 2:
        return 0.0
    (x1, y1, f1), (x2, y2, f2) = positions[-2], positions[-1]
    if f2 == f1:
        return 0.0
    pixel_distance = math.hypot(x2 - x1, y2 - y1)
    metres         = pixel_distance / PIXELS_PER_METRE
    seconds        = (f2 - f1) / fps
    return (metres / seconds) * 3.6   # m/s → km/h

print("estimate_speed() ready.")

estimate_speed() ready.


## Main Tracking Loop
Your original code is **completely unchanged**.
Speed estimation, class-aware colour coding, and violation detection are layered on top.

In [8]:
line_y_red = 430  # Red line position  ← your original, unchanged

# ── Your original counters ───────────────────────────────────────────
class_counts = defaultdict(int)
crossed_ids  = set()

# ── Added: speed tracking state ──────────────────────────────────────
track_positions = defaultdict(lambda: deque(maxlen=30))
track_speeds    = defaultdict(lambda: deque(maxlen=30))
track_classes   = {}          # remember each track's class name
violations      = []

fps       = cap.get(cv2.CAP_PROP_FPS) or 30.0
frame_idx = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLO tracking  ← YOUR LINE, unchanged
    results = model.track(frame, persist=True, classes=[1, 2, 3, 5, 6, 7])

    if results[0].boxes.data is not None:
        boxes         = results[0].boxes.xyxy.cpu()
        track_ids     = results[0].boxes.id.int().cpu().tolist()
        class_indices = results[0].boxes.cls.int().cpu().tolist()
        confidences   = results[0].boxes.conf.cpu()

        # Draw the red counting line  ← YOUR LINE, unchanged
        cv2.line(frame, (690, line_y_red), (1130, line_y_red), (0, 0, 255), 3)

        for box, track_id, class_idx, conf in zip(boxes, track_ids, class_indices, confidences):
            x1, y1, x2, y2 = map(int, box)
            cx = (x1 + x2) // 2   # ← your centre point
            cy = (y1 + y2) // 2

            class_name = class_list[class_idx]   # ← your class lookup
            track_classes[track_id] = class_name

            # ── Added: speed estimation ───────────────────────────────
            track_positions[track_id].append((cx, cy, frame_idx))

            raw_speed = estimate_speed(track_positions[track_id], fps)
            if raw_speed > 0:
                track_speeds[track_id].append(raw_speed)

            smooth_speed = (
                float(np.median(list(track_speeds[track_id])))
                if track_speeds[track_id] else 0.0
            )

            # ── Added: colour from speed zone rules ───────────────────
            box_color = get_speed_color(smooth_speed, class_name)

            # Your original centre dot  ← unchanged
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            # Your original label — extended with speed
            label = f"ID: {track_id} {class_name}  {smooth_speed:.0f} km/h"
            cv2.putText(frame, label, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

            # Your original rectangle — colour now reflects speed zone
            cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)

            # ── Added: red circle + SPEEDING tag for violations ───────
            if is_violation(smooth_speed, class_name):
                cv2.circle(frame, (cx, cy), 35, (0, 0, 255), 2)
                cv2.putText(frame, "SPEEDING", (x1, y2 + 18),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 255), 2)

            # Your original crossing logic  ← unchanged
            if cy > line_y_red and track_id not in crossed_ids:
                crossed_ids.add(track_id)
                class_counts[class_name] += 1

                # ── Added: log violation at crossing moment ────────────
                if is_violation(smooth_speed, class_name):
                    limit = SPEED_LIMITS.get(class_name, DEFAULT_SPEED_LIMIT)
                    violations.append({
                        "frame":     frame_idx,
                        "track_id":  track_id,
                        "class":     class_name,
                        "speed_kmh": round(smooth_speed, 1),
                        "limit_kmh": limit,
                    })
                    print(f"  VIOLATION! #{track_id} {class_name} "
                          f"{smooth_speed:.1f} km/h  (limit {limit} km/h)")

        # Your original class count display  ← unchanged
        y_offset = 30
        for class_name, count in class_counts.items():
            cv2.putText(frame, f"{class_name}: {count}", (50, y_offset),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            y_offset += 30

        # ── Added: TOTAL count below per-class counts ────────────────
        total_vehicles = sum(class_counts.values())
        cv2.putText(frame, f"TOTAL: {total_vehicles}", (50, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        # ── Added: violation count overlay (top-right) ────────────────
        viol_text = f"Violations: {len(violations)}"
        cv2.putText(frame, viol_text, (frame.shape[1] - 230, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        # ── Added: speed zone colour legend (bottom-left, top-down) ───
        # Anchored from bottom so all 4 rows always fit inside the frame
        legend = [
            ((255, 100, 0), " BLUE   0-10 km/h   Slow/Stopped"),
            ((0, 200, 0),   " GREEN  11-60 km/h  Safe"),
            ((0, 220, 255), " YELLOW 61-limit    Caution"),
            ((0, 0, 255),   " RED    >limit      Violation"),
        ]
        legend_start_y = frame.shape[0] - 10 - len(legend) * 22
        for i, (col, txt) in enumerate(legend):
            ly = legend_start_y + i * 22
            cv2.rectangle(frame, (8, ly - 14), (18, ly), col, -1)
            cv2.putText(frame, txt, (22, ly - 2),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (220, 220, 220), 1)

    # Your original imshow  ← unchanged
    cv2.imshow("YOLO Object Tracking & Counting", frame)

    # Your original quit key  ← unchanged
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    frame_idx += 1

# Your original cleanup  ← unchanged
cap.release()
cv2.destroyAllWindows()


0: 384x640 10 cars, 3 motorcycles, 3 buss, 1 truck, 1310.3ms
Speed: 78.2ms preprocess, 1310.3ms inference, 11.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 3 motorcycles, 3 buss, 1 truck, 1471.2ms
Speed: 5.5ms preprocess, 1471.2ms inference, 4.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 3 motorcycles, 3 buss, 1 truck, 1509.6ms
Speed: 7.2ms preprocess, 1509.6ms inference, 6.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 3 motorcycles, 3 buss, 1 truck, 1387.6ms
Speed: 4.8ms preprocess, 1387.6ms inference, 4.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 4 motorcycles, 3 buss, 1 truck, 1323.6ms
Speed: 8.2ms preprocess, 1323.6ms inference, 3.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 cars, 4 motorcycles, 3 buss, 1 truck, 1273.0ms
Speed: 4.8ms preprocess, 1273.0ms inference, 3.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 cars, 3 motorcycles, 3 

Speed: 4.5ms preprocess, 836.6ms inference, 3.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 cars, 5 motorcycles, 2 buss, 973.2ms
Speed: 3.9ms preprocess, 973.2ms inference, 3.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 5 motorcycles, 2 buss, 821.5ms
Speed: 4.4ms preprocess, 821.5ms inference, 3.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 5 motorcycles, 2 buss, 843.6ms
Speed: 3.5ms preprocess, 843.6ms inference, 3.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 5 motorcycles, 2 buss, 877.4ms
Speed: 109.8ms preprocess, 877.4ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 6 motorcycles, 2 buss, 899.5ms
Speed: 4.2ms preprocess, 899.5ms inference, 3.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 5 motorcycles, 2 buss, 1 truck, 858.7ms
Speed: 5.3ms preprocess, 858.7ms inference, 3.3ms postprocess per image at shape (1, 3, 384, 64


0: 384x640 7 cars, 8 motorcycles, 2 buss, 1 truck, 838.2ms
Speed: 4.5ms preprocess, 838.2ms inference, 3.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 8 motorcycles, 1 bus, 2 trucks, 949.6ms
Speed: 5.8ms preprocess, 949.6ms inference, 3.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 8 motorcycles, 1 bus, 2 trucks, 800.6ms
Speed: 4.7ms preprocess, 800.6ms inference, 4.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 cars, 8 motorcycles, 2 buss, 863.0ms
Speed: 5.5ms preprocess, 863.0ms inference, 3.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 8 motorcycles, 2 buss, 1 truck, 835.5ms
Speed: 4.8ms preprocess, 835.5ms inference, 3.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 8 motorcycles, 2 buss, 1 truck, 895.7ms
Speed: 5.2ms preprocess, 895.7ms inference, 3.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 8 motorcycles, 2 buss, 1 truck, 863.7ms
S

Speed: 5.4ms preprocess, 924.1ms inference, 3.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 7 motorcycles, 1 bus, 3 trucks, 878.7ms
Speed: 5.0ms preprocess, 878.7ms inference, 3.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 7 motorcycles, 2 buss, 1 truck, 827.7ms
Speed: 4.4ms preprocess, 827.7ms inference, 3.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 7 motorcycles, 2 buss, 2 trucks, 881.8ms
Speed: 4.9ms preprocess, 881.8ms inference, 3.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 7 motorcycles, 2 buss, 2 trucks, 833.2ms
Speed: 5.3ms preprocess, 833.2ms inference, 3.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 6 motorcycles, 2 buss, 1 truck, 957.3ms
Speed: 4.5ms preprocess, 957.3ms inference, 3.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 7 motorcycles, 2 buss, 1 truck, 1028.8ms
Speed: 6.2ms preprocess, 1028.8ms inference, 3.6m


0: 384x640 10 cars, 4 motorcycles, 1 bus, 3 trucks, 1099.3ms
Speed: 5.3ms preprocess, 1099.3ms inference, 3.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 4 motorcycles, 1 bus, 3 trucks, 846.8ms
Speed: 4.1ms preprocess, 846.8ms inference, 3.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 4 motorcycles, 1 bus, 3 trucks, 1002.3ms
Speed: 4.6ms preprocess, 1002.3ms inference, 3.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 4 motorcycles, 1 bus, 4 trucks, 828.7ms
Speed: 4.1ms preprocess, 828.7ms inference, 3.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 5 motorcycles, 1 bus, 4 trucks, 878.3ms
Speed: 4.4ms preprocess, 878.3ms inference, 3.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 4 motorcycles, 1 bus, 3 trucks, 906.1ms
Speed: 6.1ms preprocess, 906.1ms inference, 3.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 4 motorcycles, 1 bus, 3 t

...

0: 384x640 9 cars, 1 bus, 5 trucks, 606.9ms
Speed: 3.1ms preprocess, 606.9ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 cars, 1 bus, 2 trucks, 557.0ms
Speed: 2.7ms preprocess, 557.0ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 1 bus, 4 trucks, 585.9ms
Speed: 2.8ms preprocess, 585.9ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 1 bus, 4 trucks, 550.9ms
Speed: 2.7ms preprocess, 550.9ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 1 bus, 4 trucks, 588.4ms
Speed: 2.4ms preprocess, 588.4ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 1 bus, 3 trucks, 557.5ms
Speed: 2.6ms preprocess, 557.5ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 1 bus, 3 trucks, 573.5ms
Speed: 64.0ms preprocess, 573.5ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 64

Speed: 2.9ms preprocess, 1055.5ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 1 motorcycle, 1 bus, 2 trucks, 550.4ms
Speed: 3.5ms preprocess, 550.4ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 1 motorcycle, 1 bus, 2 trucks, 580.8ms
Speed: 2.8ms preprocess, 580.8ms inference, 2.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 1 motorcycle, 1 bus, 2 trucks, 612.4ms
Speed: 3.6ms preprocess, 612.4ms inference, 2.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 1 motorcycle, 1 bus, 2 trucks, 570.3ms
Speed: 3.5ms preprocess, 570.3ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 11 cars, 1 motorcycle, 1 bus, 2 trucks, 592.7ms
Speed: 2.9ms preprocess, 592.7ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 1 motorcycle, 1 bus, 2 trucks, 594.7ms
Speed: 3.4ms preprocess, 594.7ms inference, 2.0ms p


0: 384x640 10 cars, 2 motorcycles, 1 bus, 2 trucks, 646.5ms
Speed: 2.7ms preprocess, 646.5ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 3 motorcycles, 1 bus, 2 trucks, 670.9ms
Speed: 2.9ms preprocess, 670.9ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 2 motorcycles, 1 bus, 2 trucks, 654.8ms
Speed: 2.9ms preprocess, 654.8ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 2 motorcycles, 1 bus, 2 trucks, 670.1ms
Speed: 3.2ms preprocess, 670.1ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 11 cars, 1 motorcycle, 1 bus, 2 trucks, 654.6ms
Speed: 2.4ms preprocess, 654.6ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 11 cars, 1 motorcycle, 1 bus, 2 trucks, 581.5ms
Speed: 3.2ms preprocess, 581.5ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 11 cars, 1 motorcycle, 1 bus, 2 truc


0: 384x640 13 cars, 2 motorcycles, 1 bus, 1 truck, 643.6ms
Speed: 3.1ms preprocess, 643.6ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 cars, 2 motorcycles, 1 bus, 1 truck, 557.7ms
Speed: 2.6ms preprocess, 557.7ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 13 cars, 2 motorcycles, 1 bus, 1 truck, 570.2ms
Speed: 2.7ms preprocess, 570.2ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 cars, 2 motorcycles, 1 bus, 1 truck, 1086.1ms
Speed: 10.8ms preprocess, 1086.1ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 13 cars, 2 motorcycles, 1 bus, 1 truck, 589.2ms
Speed: 2.9ms preprocess, 589.2ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 13 cars, 2 motorcycles, 1 bus, 1 truck, 568.1ms
Speed: 2.5ms preprocess, 568.1ms inference, 2.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 cars, 2 motorcycles, 1 bus, 1 truc

KeyboardInterrupt: 

## Results Summary

In [9]:
print("=" * 58)
print("  SMART TRAFFIC ANALYSIS — CITY ROAD RESULTS")
print("=" * 58)

print()
print("VEHICLE COUNTS  (crossed red line)")
for vtype, count in class_counts.items():
    lim = SPEED_LIMITS.get(vtype, DEFAULT_SPEED_LIMIT)
    print(f"  {vtype:<14}: {count}  (speed limit {lim} km/h)")
print(f"  {'TOTAL':<14}: {sum(class_counts.values())}")

print()
print("SPEED STATISTICS")
all_speeds = [s for spds in track_speeds.values() for s in spds]
if all_speeds:
    print(f"  Mean speed   : {np.mean(all_speeds):.1f} km/h")
    print(f"  Median speed : {np.median(all_speeds):.1f} km/h")
    print(f"  Max speed    : {np.max(all_speeds):.1f} km/h")
else:
    print("  No speed data collected.")

print()
print(f"SPEED VIOLATIONS")
print(f"  Total : {len(violations)}")
for v in violations:
    print(f"    Frame {v['frame']:>5}  #{v['track_id']:<4}  "
          f"{v['class']:<12}  {v['speed_kmh']} km/h  "
          f"(limit {v['limit_kmh']} km/h)")

print()
print("COLOUR KEY  (Speed Zone Rules)")
print("  BLUE   box →  0–10 km/h   Very slow / stopped")
print("  GREEN  box → 11–60 km/h   Safe speed")
print("  YELLOW box → 61–limit     Approaching limit, caution")
print("  RED    box → above limit  Violation + SPEEDING tag + red circle")

  SMART TRAFFIC ANALYSIS — CITY ROAD RESULTS

VEHICLE COUNTS  (crossed red line)
  bus           : 8  (speed limit 40 km/h)
  motorcycle    : 15  (speed limit 50 km/h)
  car           : 35  (speed limit 50 km/h)
  truck         : 29  (speed limit 40 km/h)
  TOTAL         : 87

SPEED STATISTICS
  Mean speed   : 50.4 km/h
  Median speed : 31.8 km/h
  Max speed    : 315.9 km/h

SPEED VIOLATIONS
  Total : 10
    Frame   169  #211   motorcycle    50.3 km/h  (limit 50 km/h)
    Frame   322  #10    car           50.3 km/h  (limit 50 km/h)
    Frame   485  #239   car           50.3 km/h  (limit 50 km/h)
    Frame   624  #406   truck         47.7 km/h  (limit 40 km/h)
    Frame   695  #391   car           50.3 km/h  (limit 50 km/h)
    Frame   810  #509   car           50.3 km/h  (limit 50 km/h)
    Frame   913  #609   car           50.3 km/h  (limit 50 km/h)
    Frame  1009  #737   car           50.3 km/h  (limit 50 km/h)
    Frame  1168  #1311  motorcycle    50.3 km/h  (limit 50 km/h)
    Fra